# Menghitung jarak antara data lama(buka tutup) masing masing 100 data dengan data baru 1 data menggunakan Dynamic Time Warping



### Install library

In [1]:
!pip install dtaidistance librosa numpy tqdm

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 7.1 MB/s  0:00:00


### Import library

In [1]:
import os
import numpy as np
import librosa
import pandas as pd
from dtaidistance import dtw_ndim
from tqdm import tqdm

### CONFIG

In [3]:
DATASET_DIR = "Gabungan_Suara"   # folder utama
REKAMAN_BUKA = os.path.join(DATASET_DIR, "rekaman_baru(buka).wav")
REKAMAN_TUTUP = os.path.join(DATASET_DIR, "rekaman_baru(tutup).wav")
N_MFCC = 20
ADD_DELTAS = True     # tambahkan delta dan delta-delta
EPS = 1e-9            # untuk stabilitas pembagian

### UTIL: ekstraksi MFCC robust

In [4]:
def load_and_mfcc(path, n_mfcc=N_MFCC, trim_silence=True, add_deltas=ADD_DELTAS):
    try:
        y, sr = librosa.load(path, sr=22050)
        if trim_silence:
            y, _ = librosa.effects.trim(y, top_db=25)  # pangkas silence depan/akhir
        if len(y) < 160:  # terlalu pendek, kemungkinan error
            raise ValueError("Audio terlalu pendek setelah trim.")
        # normalisasi amplitudo sebelum MFCC
        y = librosa.util.normalize(y)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # shape: (n_mfcc, t)
        feats = mfcc
        if add_deltas:
            delta = librosa.feature.delta(mfcc)
            delta2 = librosa.feature.delta(mfcc, order=2)
            feats = np.vstack([mfcc, delta, delta2])  # shape: (n_feats, t)
        # transpose ke shape (time, features)
        feats = feats.T.astype(np.float64)
        # CMVN per-file: zero mean, unit var per coefficient (along time axis)
        mean = np.mean(feats, axis=0)
        std = np.std(feats, axis=0)
        std_adj = np.where(std < EPS, 1.0, std)
        feats = (feats - mean) / std_adj
        return feats
    except Exception as e:
        print(f"[ERROR] load_and_mfcc('{path}') -> {e}")
        return None


### LOAD DATASET (buka + tutup)

In [5]:
data_mfcc = []
labels = []
file_paths = []

for lbl in ["buka", "tutup"]:
    folder = os.path.join(DATASET_DIR, lbl)
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
    for fname in sorted(os.listdir(folder)):
        if not (fname.lower().endswith(".wav") or fname.lower().endswith(".mp3")):
            continue
        fpath = os.path.join(folder, fname)
        feats = load_and_mfcc(fpath)
        if feats is not None:
            data_mfcc.append(feats)
            labels.append(lbl)
            file_paths.append(fpath)

print(f"Loaded {len(data_mfcc)} files: buka={labels.count('buka')} tutup={labels.count('tutup')}")

if len(data_mfcc) == 0:
    raise RuntimeError("Tidak ada file audio yang berhasil dimuat.")

Loaded 100 files: buka=50 tutup=50


### Fungsi hitung jarak DTW (multidim)

In [6]:
def dtw_distance(a, b):
    # dtaidistance.dtw_ndim.distance expects arrays shape (time, dim)
    return float(dtw_ndim.distance(a, b))

### Fungsi evaluasi untuk 1 rekaman input

In [7]:
def evaluate_input(input_path, show_top_k=5, save_csv=None):
    print("\n--- Evaluasi:", input_path)
    inp = load_and_mfcc(input_path)
    if inp is None:
        print("Gagal ekstrak input.")
        return None

    distances = []
    for feats, lbl, fpath in tqdm(zip(data_mfcc, labels, file_paths), total=len(labels), desc="Computing DTW"):
        try:
            d = dtw_distance(inp, feats)
        except Exception as e:
            print(f"[WARN] DTW failed for {fpath}: {e}")
            d = np.inf
        distances.append((fpath, lbl, d))

    # DataFrame hasil
    df = pd.DataFrame(distances, columns=["file", "label", "distance"]).sort_values("distance")
    # ringkasan per-kelas
    summary = df.groupby("label")["distance"].agg(["min", "mean", "median", "count"]).reset_index()
    print("\nRingkasan per-kelas:")
    print(summary.to_string(index=False))

    # keputusan 1-NN (min overall) dan keputusan rata-rata (mean per class)
    best_overall = df.iloc[0]
    mean_per_class = df.groupby("label")["distance"].mean()
    class_by_mean = mean_per_class.idxmin()
    print("\nTop", show_top_k, "terdekat:")
    print(df.head(show_top_k).to_string(index=False))

    print("\nKeputusan berdasarkan MIN (1-NN):")
    print(f" -> file paling dekat: {best_overall['file']} (label={best_overall['label']}, dist={best_overall['distance']:.3f})")
    print("Keputusan berdasarkan MEAN (rata-rata jarak ke tiap kelas):")
    print(f" -> kelas terdekat (mean): {class_by_mean} (mean_dist={mean_per_class[class_by_mean]:.3f})")

    if save_csv:
        df.to_csv(save_csv, index=False)
        print(f"Hasil tersimpan ke {save_csv}")

    return {"df": df, "summary": summary, "min_decision": best_overall["label"], "mean_decision": class_by_mean}

### Jalankan Percobaan Buka

In [9]:
res_buka = evaluate_input(REKAMAN_BUKA, show_top_k=8, save_csv="hasil_rekaman_buka.csv")


--- Evaluasi: Gabungan_Suara\rekaman_baru(buka).wav


Computing DTW:   0%|          | 0/100 [00:00<?, ?it/s]

Computing DTW: 100%|██████████| 100/100 [00:05<00:00, 19.35it/s]


Ringkasan per-kelas:
label       min      mean    median  count
 buka 79.777685 87.424583 86.820579     50
tutup 89.545736 92.713914 91.530195     50

Top 8 terdekat:
                                file label  distance
Gabungan_Suara\buka\buka3 copy 2.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 4.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 5.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 6.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 7.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 8.wav  buka 79.777685
Gabungan_Suara\buka\buka3 copy 9.wav  buka 79.777685
  Gabungan_Suara\buka\buka3 copy.wav  buka 79.777685

Keputusan berdasarkan MIN (1-NN):
 -> file paling dekat: Gabungan_Suara\buka\buka3 copy 2.wav (label=buka, dist=79.778)
Keputusan berdasarkan MEAN (rata-rata jarak ke tiap kelas):
 -> kelas terdekat (mean): buka (mean_dist=87.425)
Hasil tersimpan ke hasil_rekaman_buka.csv


### Jalankan Percobaan Tutup

In [10]:
res_tutup = evaluate_input(REKAMAN_TUTUP, show_top_k=8, save_csv="hasil_rekaman_tutup.csv")


--- Evaluasi: Gabungan_Suara\rekaman_baru(tutup).wav


Computing DTW: 100%|██████████| 100/100 [00:04<00:00, 21.58it/s]


Ringkasan per-kelas:
label       min      mean    median  count
 buka 85.718540 90.101152 87.057281     50
tutup 84.721909 88.202942 87.336748     50

Top 8 terdekat:
                                  file label  distance
Gabungan_Suara\tutup\tutup3 copy 2.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 3.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 4.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 5.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 6.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 7.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 8.wav tutup 84.721909
Gabungan_Suara\tutup\tutup3 copy 9.wav tutup 84.721909

Keputusan berdasarkan MIN (1-NN):
 -> file paling dekat: Gabungan_Suara\tutup\tutup3 copy 2.wav (label=tutup, dist=84.722)
Keputusan berdasarkan MEAN (rata-rata jarak ke tiap kelas):
 -> kelas terdekat (mean): tutup (mean_dist=88.203)
Hasil tersimpan ke hasil_rekaman_tutup.csv


### Ringkasan akhir

In [11]:
# Ringkasan akhir
print("\n=== RINGKASAN AKHIR ===")
if res_buka is not None:
    print("rekaman_baru(buka).wav -> MIN:", res_buka["min_decision"], "| MEAN:", res_buka["mean_decision"])
if res_tutup is not None:
    print("rekaman_baru(tutup).wav -> MIN:", res_tutup["min_decision"], "| MEAN:", res_tutup["mean_decision"])


=== RINGKASAN AKHIR ===
rekaman_baru(buka).wav -> MIN: buka | MEAN: buka
rekaman_baru(tutup).wav -> MIN: tutup | MEAN: tutup
